In [1]:
import deeplabcut, shutil
import pandas as pd
from pathlib import Path
from deeplabcut.utils import auxiliaryfunctions

DATA = Path(r".\data")
PROJECT = next(Path(r".\dlc").glob("pupil-dani-*"))
config_path = str(PROJECT / "config.yaml")
SESSIONS = [
    "094124",
    "110535",
    "115300",
    "131707",
    "135059",
    "171101",
    "172731",
    "214411",
]
TEST_SESSION = "110535"
print(config_path)

dlc\pupil-dani-2026-11-13\config.yaml


In [2]:
def configure_project(config_path, sessions, frame_size=(1280, 1024)):
    cfg = auxiliaryfunctions.read_config(config_path)
    w, h = frame_size
    proj = cfg["project_path"]
    cfg["bodyparts"] = [
        "eye-anterior",
        "eye-posterior",
        "eye-dorsal",
        "eye-ventral",
        "pupil-anterior",
        "pupil-posterior",
        "pupil-dorsal",
        "pupil-ventral",
    ]
    cfg["video_sets"] = {
        f"{proj}/videos/{s}.mp4": {"crop": f"0, {w}, 0, {h}"} for s in sessions
    }
    cfg["pcutoff"] = 0.6
    cfg["default_net_type"] = "resnet_50"
    cfg["TrainingFraction"] = [0.9]
    auxiliaryfunctions.write_config(config_path, cfg)

configure_project(config_path, SESSIONS)

In [3]:
def copy_labeled_data(src_root, config_path, sessions):
    proj = Path(auxiliaryfunctions.read_config(config_path)["project_path"])
    for s in sessions:
        src, dst = Path(src_root) / "labeled-data" / s, proj / "labeled-data" / s
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            shutil.copy2(f, dst / f.name)


copy_labeled_data(DATA, config_path, SESSIONS)

In [ ]:
deeplabcut.create_training_dataset(config_path)


def merged_counts(config_path):
    cfg = auxiliaryfunctions.read_config(config_path)
    folder = Path(cfg["project_path"]) / auxiliaryfunctions.get_training_set_folder(cfg)
    df = pd.read_hdf(next(folder.glob("CollectedData_*.h5")))
    counts = pd.Series([i[1] for i in df.index]).value_counts().sort_index()
    print(counts.to_string())
    print("total frames:", len(df))
    return df


df = merged_counts(config_path)

094124    40
110535    39
115300    34
131707    60
135059    40
171101    20
172731    80
214411    80
total frames: 393


In [6]:
def split_by_session(config_path, test_session):
    cfg = auxiliaryfunctions.read_config(config_path)
    folder = Path(cfg["project_path"]) / auxiliaryfunctions.get_training_set_folder(cfg)
    df = pd.read_hdf(next(folder.glob("CollectedData_*.h5")))
    test = [i for i, idx in enumerate(df.index) if idx[1] == test_session]
    train = [i for i, idx in enumerate(df.index) if idx[1] != test_session]
    return train, test


train_inds, test_inds = split_by_session(config_path, TEST_SESSION)
print(f"train frames: {len(train_inds)} | test frames: {len(test_inds)}")

deeplabcut.create_training_dataset(
    config_path,
    Shuffles=[1],
    trainIndices=[train_inds],
    testIndices=[test_inds],
    net_type="resnet_50",
)

train frames: 354 | test frames: 39
You passed a split with the following fraction: 90%


[(0.9,
  1,
  (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
           13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
           26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
           39,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
           91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
          104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
          117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
          130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
          143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
          156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
          169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,
          182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194,
          195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207,
          208

In [ ]:
deeplabcut.train_network(
    config_path, shuffle=1, saveiters=5000, displayiters=500, maxiters=50000
)